In [ ]:
import numpy as np
import torch
import sys
import os
from torch.utils.data import get_worker_info
from torch.utils.data import DataLoader,IterableDataset
project_root = "/mnt/sunxh/sunxh/wattmamba"
sys.path.insert(0, project_root)
os.chdir(project_root)
print(os.getcwd())
from model.model import WaveCrossMamba,AnomalyDetectionModel   
kmer_encode_dic={'A': 0, "C": 1, "G": 2, "T": 3}

class PredictIterableDataset(IterableDataset):
    def __init__(self, file_path):
        super(PredictIterableDataset).__init__()
        self.file_path = file_path
    def parse_line(self, line):
        items = line.strip().split("\t")
        if len(items) < 14:
            return None
        try:
            read_id = items[0]
            contig = items[1]
            position = items[2]
            motif = items[3]

            signal = np.array([float(x) for x in "|".join(items[9:14]).split("|")])
            kmer = np.array([kmer_encode_dic[base] for base in motif])
            mean = np.array([float(x) for x in items[4].split("|")])
            std = np.array([float(x) for x in items[5].split("|")])
            intense = np.array([float(x) for x in items[6].split("|")])
            dwell = np.array([float(x) for x in items[7].split("|")]) / 200.0
            base_quality = np.array([float(x) for x in items[8].split("|")]) / 40.0
            x = [
                torch.tensor(signal, dtype=torch.float32).unsqueeze(0).unsqueeze(2),
                torch.tensor(kmer, dtype=torch.long),
                torch.tensor(mean, dtype=torch.float32),
                torch.tensor(std, dtype=torch.float32),
                torch.tensor(intense, dtype=torch.float32),
                torch.tensor(dwell, dtype=torch.float32),
                torch.tensor(base_quality, dtype=torch.float32),
            ]
            y = "|".join([contig, position, motif, read_id])
            return x, y
        except Exception as e:
            return None


    def __iter__(self):
        info = get_worker_info()
        if info is None:
            worker_id, num_workers = 0, 1
        else:
            worker_id, num_workers = info.id, info.num_workers

        with open(self.file_path, "r") as f:
            for i, line in enumerate(f):
                if i % num_workers != worker_id:
                    continue
                result = self.parse_line(line)
                if result is not None:
                    yield result

def predict_model(pretrain_model, model, test_loader, device, output_predict, max_write=None):
    with open(output_predict, "w") as predict_result:
        model.to(device)
        pretrain_model.to(device)
        model.eval()
        pretrain_model.eval()

        label_dict = {0: "unmod", 1: "mod"}
        written = 0

        with torch.no_grad():
            for batch_idx, (data, batch_y) in enumerate(test_loader):
                if max_write is not None and written >= max_write:
                    break

                x, kmer, mean, std, intense, dwell, base_quality = data
                x = x.to(device)
                kmer = kmer.to(device)
                mean = mean.to(device)
                std = std.to(device)
                intense = intense.to(device)
                dwell = dwell.to(device)
                base_quality = base_quality.to(device)

                logits, ff = pretrain_model(x, kmer, mean, std, intense, dwell, base_quality)
                out = model(logits)

                out = torch.softmax(out, dim=1)
                pred = torch.max(out, 1)[1].cpu().numpy()
                probabilities = out.detach().cpu().numpy()[:, 1]

                bs = len(batch_y)
                take = bs if max_write is None else min(bs, max_write - written)

                for j in range(take):
                    label_str = batch_y[j]
                    parts = label_str.split("|")
                    position, motif, read_id = parts[-3:]
                    contig_full = "|".join(parts[:-3])
                    contig = contig_full.split("|", 1)[0]

                    print("%s\t%s\t%s\t%s\t%s\t%s" % (
                        contig,
                        position,
                        motif,
                        read_id,
                        label_dict[int(pred[j])],
                        float(probabilities[j])
                    ), file=predict_result)

                written += take
                if (written % 1_000_000) < take:
                    print(f"[INFO]  {written} ")

        return written

import gc, torch

def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_dataset = PredictIterableDataset(args.input)
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=10000, 
        num_workers=4,
        pin_memory=True
    )

    pretrained_feature_extractor  = WaveCrossMamba(device=device,d_model=128).to(device)
    fine_tune_model  = AnomalyDetectionModel(feature_dim=128, num_classes=2).to(device)
    pretrained_feature_extractor.load_state_dict(torch.load(args.pre,map_location=device))
    fine_tune_model.load_state_dict(torch.load(args.fine, map_location=device))
    max_n = getattr(args, "max_n", 8_000_000)

    n_written = predict_model(pretrained_feature_extractor, fine_tune_model, test_loader, device, args.o, max_write=max_n)
    del pretrained_feature_extractor, fine_tune_model, test_loader, test_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    

/mnt/sunxh/sunxh/wattmamba


In [ ]:
import os
from types import SimpleNamespace

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import gc, torch

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/save_model/m6a_model.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/save_model/m6a_classier.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nnann.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.m6a.tsv',
    gpu='0',
)
main(args)


测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条
[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.m6a.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.m6a.tsv


In [3]:
args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/save_model/m1a_model.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/save_model/m1a_classier.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nnann.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.m1a.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条


[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.m1a.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.m1a.tsv


In [4]:

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/base_a/pre_inosine_two.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/base_a/fine_inosine_two.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nnann.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.Inosine.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条


[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.Inosine.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.Inosine.tsv


In [5]:

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/backdone_f5c.pth",
    fine ="/mnt/sunxh/sunxh/wattmamba/save_result/other_base/classierf5c.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nncnn.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.f5c.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条
[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.f5c.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.f5c.tsv


In [ ]:

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/backdone_hm5C.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/classierhm5C.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nncnn.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.hm5c.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条
[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.hm5c.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.hm5c.tsv


In [7]:

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/save_model/m5c_model.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/save_model/m5c_classier.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nncnn.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.m5c.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条
[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.m5c.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.m5c.tsv


In [8]:

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/backdone_m5u.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/classier_m5u.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nntnn.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.m5u.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条
[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.m5u.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.m5u.tsv


In [9]:

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/backdone_m1y.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/classier_m1y.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nntnn.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.m1y.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条
[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.m1y.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.m1y.tsv


In [10]:

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/pre_psu_two.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/fine_psu_two.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nntnn.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.psu.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条
[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条
[INFO] 已写入 8000000 条
预测完成，实际写入 8000000 条：/mnt/sunxh/sunxh/project/HEYA8/rep1.psu.tsv
预测完成，生成文件/mnt/sunxh/sunxh/project/HEYA8/rep1.psu.tsv


In [11]:

args = SimpleNamespace(
    pre = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/pre_m7g_two.pth",
    fine = "/mnt/sunxh/sunxh/wattmamba/save_result/other_base/fine_m7g_two.pth",
    input='/mnt/sunxh/sunxh/project/HEYA8/rep1.nngnn.tsv',
    o='/mnt/sunxh/sunxh/project/HEYA8/rep1.m7g.tsv',
    gpu='0',
)
main(args)

测试数据加载完成。
开始评估二分类异常检测模型...
开始预测，最多写入 8000000 条...
[INFO] 已写入 1000000 条
[INFO] 已写入 2000000 条
[INFO] 已写入 3000000 条
[INFO] 已写入 4000000 条
[INFO] 已写入 5000000 条
[INFO] 已写入 6000000 条
[INFO] 已写入 7000000 条


RuntimeError: Caught RuntimeError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/mnt/sunxh/miniconda3/envs/mamba/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 308, in _worker_loop
    data = fetcher.fetch(index)
  File "/mnt/sunxh/miniconda3/envs/mamba/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 42, in fetch
    return self.collate_fn(data)
  File "/mnt/sunxh/miniconda3/envs/mamba/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 265, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
  File "/mnt/sunxh/miniconda3/envs/mamba/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 142, in collate
    return [collate(samples, collate_fn_map=collate_fn_map) for samples in transposed]  # Backwards compatibility.
  File "/mnt/sunxh/miniconda3/envs/mamba/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 142, in <listc